# Quantization

A standard pretrained LLM stores its parameters as 32-bit or 16-bit floating point numbers.  An 8-billion-parameter model in fp16 needs roughly 16 GB of memory just to hold the weights — too much for a typical laptop, and often too much even for a single consumer GPU.

**Quantization** is the process of replacing those high-precision numbers with low-precision approximations.  Common targets are:

* 8-bit integers (`Q8`): about half the memory of fp16
* 4-bit integers (`Q4`): about one-quarter the memory of fp16
* sometimes 3-bit or even 2-bit (`Q3`, `Q2`) for extreme compression

The trade-off is accuracy.  Aggressive quantization can degrade output quality, but in practice the loss at 4-bit is surprisingly small.  That's why 4-bit quantized models have become the de facto standard for running LLMs locally.

In this notebook we will:

* see mechanically what quantization does to a single tensor,
* survey the quantization labels (`Q4_K_M`, `IQ3_M`, ...) you'll find on Hugging Face,
* load a pre-quantized GGUF model and run inference with `llama-cpp-python`,
* quantize a full-precision Hugging Face model on the fly with `bitsandbytes`,
* benchmark full-precision against 4-bit to see the actual memory and speed differences.

## Quantizing a tensor by hand

Before reaching for any library, it helps to see what quantization actually does to a chunk of numbers.  We'll take a small tensor of "weights" and quantize it to 4-bit integers using the simplest scheme, symmetric uniform quantization:

1. Find the largest absolute value in the tensor.
2. Choose a *scale* that maps that value to the largest representable integer.
3. Divide every weight by the scale and round to the nearest integer.
4. To use the weight, multiply the integer back by the scale (this is called dequantization).

The original full-precision number is gone — what's stored is just a small integer plus one shared scale per group of weights.

In [ ]:
import torch

Pretend these eight numbers are the weights of one tiny layer:

In [ ]:
weights = torch.tensor([-0.42, 0.13, 0.81, -0.05, 0.27, -0.66, 0.51, -0.30])

Say that we want to store these as 4-bit signed integers.  A signed 4-bit integer can represent values in the range `[-7, 7]` (we leave -8 out so the range is symmetric around zero).  We need a scale factor that maps the largest absolute weight to 7:

In [ ]:
max_abs = weights.abs().max()
max_abs

In [ ]:
scale = max_abs / 7
scale

Now divide every weight by the scale and round to the nearest integer:

In [ ]:
q = torch.round(weights / scale).to(torch.int8)
q

These eight small integers (plus the one shared `scale`) are all we need to store.  To use the weights — for example to compute a matrix multiplication — we have to dequantize them back to floats:

In [ ]:
dequantized = q.to(torch.float32) * scale
dequantized

Compare with the original.  The reconstructed values are close but not identical — every weight has picked up a small rounding error:

In [ ]:
for orig, deq in zip(weights, dequantized):
    print(f"original: {orig.item(): .4f}   dequantized: {deq.item(): .4f}   error: {(orig - deq).item(): .4f}")

How much memory did we save?  
* The original tensor stored 8 floats x 32 bits each = 256 bits.
* The quantized version stores 8 integers x 4 bits = 32 bits.
* This gives the 8x compression, the ratio we'd expect going from fp32 to 4-bit.
* There is extra contribution from the scale factor, which might be a 16- or 32-bit number.

Wrapping this up into a function so we can reuse it:

In [ ]:
def quantize_4bit_symmetric(w):
    """Symmetric 4-bit quantization. Returns (int8 codes, fp scale)."""
    scale = w.abs().max() / 7
    q = torch.round(w / scale).to(torch.int8)
    return q, scale

def dequantize_4bit_symmetric(q, scale):
    return q.to(torch.float32) * scale

In [ ]:
q, s = quantize_4bit_symmetric(weights)
q, s

In [ ]:
dequantize_4bit_symmetric(q, s)

Real quantization schemes (used by `llama.cpp`, `bitsandbytes`, GPTQ, AWQ, etc.) are more sophisticated than this:

* they apply a separate scale to small groups of weights (e.g. every 32 or 64 weights), so a single outlier doesn't blow up the scale for an entire layer,
* some include a learned zero point so the integer range doesn't have to be symmetric around zero,
* some use non-uniform spacings tuned to the actual distribution of weights (`NF4` — NormalFloat-4 — is built for weights that look roughly normally distributed).

But the basic idea — store small integers, keep a scale around, dequantize on the fly — is exactly what we just did.

## GGUF quantization labels

The most common file format for distributing quantized LLMs is GGUF (used by `llama.cpp` and `llama-cpp-python`).  Quantized GGUF files come with cryptic labels like `Q4_K_M`, `Q5_K_S`, or `IQ3_M`.  These encode:

* the bit width (`Q4` = 4-bit, `Q5` = 5-bit, ...),
* the quantization variant (`K` = a newer "K-quant" scheme with mixed precision across layers; `I` prefix = "importance" quants that use a calibration dataset),
* the size tier within that variant (`S` = small, `M` = medium, `L` = large).

`Q4_K_M` (4-bit, K-quant, medium) tends to be the sweet spot for most users — small enough to fit on a laptop, accurate enough to be hard to distinguish from the full-precision model in casual use.

To find a quantized model on Hugging Face:

* Go to Models -> Libraries -> GGUF and sort by trending or downloads.
* Prefer trusted quantizers like bartowski, unsloth, or TheBloke.
* For chat use, pick a build whose name contains `Instruct`, `Chat`, or `Reasoning`.

The example model used below comes from <https://huggingface.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF>, which has a useful "Which file should I choose?" section.  For a deeper dive into label meanings, see <https://gist.github.com/Artefact2/b5f810600771265fc1e39442288e8ec9>.

### Downloading a GGUF file

The current Hugging Face CLI uses `hf download` (the older `huggingface-cli download` form still works for now but is being phased out — the libraries are changing fast).

Two example commands.  The first grabs a 4-bit quantized build; the second grabs the full-precision fp16 version of the same model so we can compare them later:

In [ ]:
# These will download files to your local directory
# Uncomment them if needed
#
# !hf download bartowski/Llama-3.2-1B-Instruct-GGUF --include "Llama-3.2-1B-Instruct-Q4_K_M.gguf" --local-dir ./
# !hf download bartowski/Llama-3.2-1B-Instruct-GGUF --include "Llama-3.2-1B-Instruct-f16.gguf" --local-dir ./

## Loading a pre-quantized GGUF model

With a GGUF file in hand we can load and run it through `llama-cpp-python`.  This library runs on CPU (and optionally GPU), which is why the quantized formats matter so much — a 4-bit model is small and fast enough to run on a laptop without a GPU.

In [ ]:
from llama_cpp import Llama

Point `Llama` at the path to the downloaded `.gguf` file.  `n_ctx` is the context window in tokens; `n_threads` should be roughly the number of physical CPU cores.

In [ ]:
llm = Llama(
    model_path="Llama-3.2-1B-Instruct-Q4_K_M.gguf",
    n_ctx=2048,
    n_threads=4,
    verbose=False
)

Now we can prompt the model and get a completion back:

In [ ]:
prompt = "Explain what a quantized language model is in one short paragraph."

result = llm(
    prompt,
    max_tokens=128,
    temperature=0.7,
)

print(result["choices"][0]["text"])

## Skipping the manual download

If you'd rather not download the GGUF file separately, `Llama.from_pretrained` will fetch it from Hugging Face the first time you call it and cache it locally:

In [ ]:
llm = Llama.from_pretrained(
    repo_id="bartowski/Llama-3.2-1B-Instruct-GGUF",
    filename="Llama-3.2-1B-Instruct-Q4_K_M.gguf",
    verbose=False,
    n_threads=4,
)

## Two ways to call the model

The model exposes both a raw-completion interface and a chat-style interface.

The **raw completion** treats the prompt as text to continue.  This is how the model was originally pretrained, and it works fine for short factual prompts:

In [ ]:
llm("What is the capital of France?")

The **chat completion** wraps the prompt in the model's official chat template (system/user/assistant turns).  For instruction-tuned models this usually gives cleaner answers and lets you steer behavior with a system message:

In [ ]:
llm.create_chat_completion(
    messages=[
        {"role": "user", "content": "What is the capital of France?"},
    ],
)

Adding a system message to control the response style:

In [ ]:
llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "Use one word answers."},
        {"role": "user",   "content": "What is the capital of France?"},
    ],
)

## Quantizing at load time with `transformers` + `bitsandbytes`

The GGUF workflow uses files that were quantized ahead of time, by someone else.  If you instead want to download a *full-precision* Hugging Face model and quantize it yourself at load time, the standard tool is **`bitsandbytes`**, used through the `transformers` library.  This path requires a CUDA GPU.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

Pick a model.  Llama 3.2 1B Instruct is small enough to fit comfortably on most GPUs even in full precision, which makes it a good candidate for comparing quantized vs unquantized.

In [ ]:
model_id = "meta-llama/Llama-3.2-1B-Instruct"

`BitsAndBytesConfig` tells `transformers` how to quantize the weights as they're loaded:

* `load_in_4bit=True` — store weights as 4-bit values rather than fp16.
* `bnb_4bit_quant_type="nf4"` — use NormalFloat-4, a non-uniform code designed for normally distributed weights.  The other option is `"fp4"`.
* `bnb_4bit_use_double_quant=True` — also quantize the per-group scales, for a bit of extra memory savings.
* `bnb_4bit_compute_dtype="bfloat16"` — math during inference is done in bfloat16; only storage is 4-bit.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="bfloat16",
)

Llama 3 is a gated model on Hugging Face: you have to accept the license on the model page and then use an access token to download it.  Create the token at <https://huggingface.co/settings/tokens> and store it in a local `keys.py` (don't commit it!).  Without the token you'll get a download error that isn't always obvious about what's missing.

In [ ]:
import keys
tokenizer = AutoTokenizer.from_pretrained(model_id, token=keys.HF_TOKEN)

Load the model with the quantization config attached.  `device_map="auto"` lets `accelerate` place weights on whatever GPU(s) are available:

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=keys.HF_TOKEN,
)

A quick generation to confirm it works:

In [ ]:
prompt = "Explain in one sentence what a quantized language model is."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=64,
    do_sample=True,
    temperature=0.7,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Benchmark: full-precision vs quantized (GGUF)

The whole point of quantization is to trade a small amount of accuracy for a large amount of memory and (often) speed.  Let's measure those gains directly by running the same prompts through both an fp16 GGUF model and a 4-bit GGUF model.

We'll measure three things: file size on disk (a proxy for memory use), load time, and per-prompt generation time.

In [ ]:
import os
import time
from llama_cpp import Llama

Paths to the two GGUF files we downloaded earlier:

In [ ]:
FULL_PRECISION_MODEL_PATH = "Llama-3.2-1B-Instruct-f16.gguf"
QUANTIZED_MODEL_PATH = "Llama-3.2-1B-Instruct-Q4_K_M.gguf"

A small set of prompts to run through each model.  Keeping them short keeps the benchmark fast:

In [ ]:
PROMPTS = [
    "Explain what a large language model is in 3 short bullet points.",
    "Give me 3 simple project ideas for learning Python.",
    "In a short paragraph, describe the tradeoffs between full-precision and quantized models.",
    "Summarize what overfitting is in machine learning in 2-3 sentences.",
]

The benchmark function loads a model, runs every prompt through it, and reports timing.  We bundle the chat-style wrapping inline so the test is self-contained.

In [ ]:
def benchmark_model(model_path, label):
    print("\n====================================")
    print(label)
    print("====================================")

    # File size on disk -- a rough proxy for memory footprint.
    file_size_mb = os.path.getsize(model_path) / (1024 * 1024)
    print(f"Model file size: {file_size_mb:.1f} MB")

    # Load the model and time it.
    start_load = time.time()
    llm = Llama(model_path=model_path, n_ctx=2048, n_threads=4, verbose=False)
    load_time = time.time() - start_load
    print(f"Model load time: {load_time:.1f} seconds")

    gen_times = []
    for i, user_prompt in enumerate(PROMPTS, start=1):
        print("\n------------------------------------")
        print(f"Prompt {i}: {user_prompt}")

        full_prompt = (
            "You are a helpful AI assistant.\n\n"
            f"User: {user_prompt}\n"
            "Assistant:"
        )

        start_gen = time.time()
        result = llm(full_prompt, max_tokens=160, temperature=0.2)
        gen_time = time.time() - start_gen
        gen_times.append(gen_time)

        answer = result["choices"][0]["text"].strip()
        print(f"Time for this prompt: {gen_time:.1f} seconds")
        print("Answer preview (first 300 characters):")
        print(answer[:300])

    avg_time = sum(gen_times) / len(gen_times)
    print(f"\nAverage generation time over {len(PROMPTS)} prompts: {avg_time:.1f} seconds")

Run the benchmark on both models and compare:

In [ ]:
benchmark_model(FULL_PRECISION_MODEL_PATH, "FULL PRECISION MODEL (F16)")
benchmark_model(QUANTIZED_MODEL_PATH, "QUANTIZED MODEL (Q4_K_M, 4-bit)")

The quantized file is roughly 3-4x smaller on disk, loads in comparable time, and generates tokens at a comparable or faster rate per prompt.  The answers themselves should look qualitatively similar — that's the central claim of 4-bit quantization.

## Benchmark: full-precision vs 4-bit (`transformers` + `bitsandbytes`)

The same comparison on the GPU side: load the Hugging Face model once in fp16, once in 4-bit via `bitsandbytes`, and run the same prompts through each.

In [ ]:
import keys
import time
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

In [ ]:
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

A slightly larger and more varied prompt set to stress-test both models (arithmetic, multi-step instructions, code, reasoning, story-writing):

In [ ]:
PROMPTS = [
    "You have 17 apples and give 9 to your friend, then buy 4 more. "
    "How many apples do you have now? Explain your reasoning step by step.",

    "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
    "How much does the ball cost? Explain your reasoning carefully.",

    "First, list 5 different animals. Second, sort them in alphabetical order. "
    "Third, output them as a numbered list. Do each step in order and clearly label them.",

    "Write a small Python function called is_palindrome(text) that returns True if text "
    "is a palindrome and False otherwise. Do not use Python slicing like text[::-1]. "
    "Then show what is_palindrome('racecar') and is_palindrome('hello') would return.",

    "Explain in simple terms how a search engine works, in 4-6 short steps, "
    "starting from when a user types a query and presses Enter.",

    "In 3-4 sentences, explain what 'quantization' means for neural networks, "
    "and why someone might use 4-bit weights instead of 16-bit weights.",

    "Write a short story (4-6 sentences) that includes: "
    "(1) a cat, (2) a broken robot, and (3) a surprise happy ending. "
    "At the end, add a one-sentence moral of the story.",

    "Three boxes are labeled 'apples', 'oranges', and 'apples & oranges'. "
    "You know that every label is wrong. You may pick 1 fruit from 1 box. "
    "Describe a strategy to relabel the boxes correctly and explain why it works.",
]

Two loader functions, one per precision level.  Both return `(tokenizer, model, load_time)` so we can compare load times directly:

In [ ]:
def load_full_precision_model():
    print("\n=== Loading FULL PRECISION model (FP16) ===")
    print(f"Pre-load GPU memory consumption {torch.cuda.memory_allocated(0) / (1024**3)}")
    start = time.time()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=keys.HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        token=keys.HF_TOKEN,
    )
    load_time = time.time() - start
    print(f"Full precision model loaded in {load_time:.1f} seconds")
    print(f"Post-load GPU memory consumption {torch.cuda.memory_allocated(0) / (1024**3)}")
    return tokenizer, model, load_time

In [ ]:
def load_4bit_quantized_model():
    print("\n=== Loading 4-BIT QUANTIZED model (bitsandbytes) ===")
    print(f"Pre-load GPU memory consumption {torch.cuda.memory_allocated(0) / (1024**3)}")
    start = time.time()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=keys.HF_TOKEN)

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=quant_config,
        device_map="auto",
        token=keys.HF_TOKEN,
    )
    load_time = time.time() - start
    print(f"4-bit quantized model loaded in {load_time:.1f} seconds")
    print(f"Post-load GPU memory consumption {torch.cuda.memory_allocated(0) / (1024**3)}")
    return tokenizer, model, load_time

And a benchmark function that runs every prompt through a given (tokenizer, model) pair and times generation:

In [ ]:
def run_benchmark(tokenizer, model, label):
    print(f"\n============================")
    print(label)
    print(f"============================")

    gen_times = []
    for i, user_prompt in enumerate(PROMPTS, start=1):
        print("\n------------------------------------")
        print(f"Prompt {i}: {user_prompt}")

        full_prompt = (
            "You are a helpful AI assistant.\n\n"
            f"User: {user_prompt}\n"
            "Assistant:"
        )

        inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)

        start = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.2,
                do_sample=True,
            )
        gen_time = time.time() - start
        gen_times.append(gen_time)

        full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if "Assistant:" in full_output:
            answer = full_output.split("Assistant:", 1)[1].strip()
        else:
            answer = full_output.strip()

        print(f"Time for this prompt: {gen_time:.1f} seconds")
        print("Answer preview (first 300 characters):")
        print(answer[:300])

    avg_time = sum(gen_times) / len(gen_times)
    print(f"\nAverage generation time over {len(PROMPTS)} prompts: {avg_time:.1f} seconds")

Run both:

In [ ]:
if not torch.cuda.is_available():
    print("WARNING: No GPU detected. This benchmark is designed for a CUDA GPU "
          "and will be extremely slow on CPU.")

# Full precision
tok_fp, model_fp, fp_load_time = load_full_precision_model()
run_benchmark(tok_fp, model_fp, f"FULL PRECISION (FP16)  | Load time: {fp_load_time:.1f}s")

# 4-bit quantized
tok_4bit, model_4bit, q_load_time = load_4bit_quantized_model()
run_benchmark(tok_4bit, model_4bit, f"4-BIT QUANTIZED (bitsandbytes) | Load time: {q_load_time:.1f}s")

The 4-bit model should use roughly 1/4 the GPU memory of the fp16 model, with answer quality that is hard to distinguish on prompts of this difficulty.  The 4-bit version will actually consume more memory than this, due to overhead.  It may also load slower (because the quantization step itself takes time) and may also be slower at inference (due to weight dequantization).  For faster inference, one can use pre-quantized models and optimized inference engines that support native low-bit compute kernels.